# 🌊 Challenge A — Resilient Release Scheduling: Embalse Falcón
## Hackathon Latinoamérica Cuántico 2026 · QCentroid GPU Platform

---

**Objetivo:** Maximizar el Storage Resilience Score (SRS) del Embalse Internacional Falcón mediante optimización híbrida cuántico-clásica.

**Pipeline:**
```
Datos IBWC → Baselines (Histórico + Regla) → Genético → QUBO + SA(GPU) → Benchmark ΔSRS
```

**Instancia oficial:** T=26 semanas, L=5 niveles discretos → 5²⁶ ≈ 1.49×10¹⁸ combinaciones

## Celda 0 — Instalación de dependencias

In [ ]:
import sys, subprocess

# Dependencias base (siempre)
pkgs_base = ['pandas', 'numpy', 'scikit-learn', 'deap', 'scipy', 'joblib', 'matplotlib', 'openpyxl', 'ipywidgets']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs_base)

# GPU (solo si estamos en QCentroid o máquina con NVIDIA)
try:
    import cupy
    print('✅ CuPy ya instalado')
except ImportError:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cupy-cuda12x'])
        print('✅ CuPy instalado (CUDA 12)')
    except Exception:
        print('⚠️  CuPy no disponible — se usará modo CPU')

# CUDA-Q (solo QCentroid)
try:
    import cudaq
    print('✅ CUDA-Q ya instalado')
except ImportError:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cudaq'])
        print('✅ CUDA-Q instalado')
    except Exception:
        print('⚠️  CUDA-Q no disponible — QAOA desactivado')

print('\n✅ Dependencias listas')


## Celda 1 — Imports y configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, time, os, json
warnings.filterwarnings('ignore')
try:
    from scipy.sparse import SparseEfficiencyWarning
    warnings.filterwarnings('ignore', category=SparseEfficiencyWarning)
except Exception:
    pass

# Carpeta de salidas: crearla desde el inicio para evitar FileNotFoundError.
os.makedirs('resultados', exist_ok=True)
print('📁 Carpeta de salida lista: resultados/')

# Detección de GPU
try:
    import cupy as cp
    CUPY = True
    print(f'🔥 CuPy disponible — GPU: {cp.cuda.runtime.getDeviceProperties(0)["name"].decode()}')
except ImportError:
    CUPY = False
    print('💻 Sin GPU — modo CPU')

try:
    import cudaq
    CUDAQ = True
    print('⚛️  CUDA-Q disponible')
except ImportError:
    CUDAQ = False
    print('ℹ️  CUDA-Q no disponible (QAOA desactivado)')

# ── Constantes globales ──────────────────────────────────────────────────────
S_MAX   = 3_387_000.0   # TCM — Capacidad total Falcón
NAMO    = 3_265_400.0   # TCM — Nivel de Aguas Máximas de Operación
S_MIN   = 0.25 * S_MAX  # TCM — Umbral crítico (25% NAMO)
L       = 5             # Niveles discretos de ajuste
A_VEC   = np.array([-2, -1, 0, 1, 2], dtype=float)
T       = 26            # Semanas — instancia Medium oficial
FECHA   = '2024-01-01'

print(f'\n📐 Parámetros: T={T} sem, L={L} niveles, S_MAX={S_MAX/1e6:.3f}M TCM')
print(f'   S_MIN (crítico 25%) = {S_MIN:,.0f} TCM')


## Celda 2 — Carga de datos IBWC

In [ ]:
from genetico_v3 import preparar_ventana_semanal, calcular_srs

print('Leyendo datos IBWC...')
df_lib    = pd.read_excel('R_observ.xlsx')
df_cambio = pd.read_csv(
    'DataSetExport-Discharge Total.Change-in-Storage@08461200'
    '-Instantaneous-TCM-20260622185956.csv', skiprows=1)
df_total  = pd.read_csv(
    'DataSetExport-Total Storage.Web-Daily-ac-ft@08461200'
    '-Instantaneous-TCM-20260622185130.csv', skiprows=1)
df_evap   = pd.DataFrame(columns=['Timestamp (UTC-06:00)', 'Evaporacion_mm'])
df_bat    = pd.read_csv('tabla_elevacion_volumen_FINAL.csv')

R_obs, Delta_S_obs, S_inicial = preparar_ventana_semanal(
    df_lib, df_cambio, df_total, df_evap, df_bat, FECHA, T)

delta_u = 0.25 * float(np.median(R_obs))
u_max   = 2 * delta_u
NIVELES = A_VEC * delta_u

print(f'✅ Ventana cargada: T={len(R_obs)} semanas desde {FECHA}')
print(f'   S_inicial = {S_inicial:,.0f} TCM  ({S_inicial/NAMO*100:.1f}% NAMO)')
print(f'   R_obs media = {R_obs.mean():,.0f} TCM/sem')
print(f'   Δu = {delta_u:,.0f} TCM  |  u_max = {u_max:,.0f} TCM')
print(f'   Niveles: {np.round(NIVELES, 0)}')

## Celda 3 — Baselines (Histórico y Regla de Umbral)

In [ ]:
# ── Baseline 0: Histórico (u=0 siempre) ─────────────────────────────────────
u_hist = np.zeros(T)
srs_hist, S_hist_traj = calcular_srs(u_hist, R_obs, Delta_S_obs, S_inicial, S_MAX)
print(f'Baseline 0 — Histórico    SRS = {srs_hist:.6f}')

# ── Baseline 1: Regla de umbral (u=-Δu si S<S_MIN, si no u=0) ───────────────
u_rule = np.zeros(T)
S_r    = S_inicial
for t in range(T):
    u_rule[t] = -delta_u if S_r < S_MIN else 0.0
    S_r += Delta_S_obs[t] - u_rule[t]

srs_rule, S_rule_traj = calcular_srs(u_rule, R_obs, Delta_S_obs, S_inicial, S_MAX)
print(f'Baseline 1 — Regla umbral SRS = {srs_rule:.6f}  ΔSRS = {srs_rule-srs_hist:+.6f}')

resultados = {
    'Baseline 0 — Histórico (u=0)': {'srs': srs_hist, 'u': u_hist, 'S_traj': S_hist_traj},
    'Baseline 1 — Regla umbral':    {'srs': srs_rule, 'u': u_rule, 'S_traj': S_rule_traj},
}

## Celda 4 — Baseline 2: Algoritmo Genético (DEAP)

In [ ]:
from genetico_v3 import run_genetico

print('⚙️  Ejecutando Genético (n_pop=500, n_gen=200)...')
t0 = time.time()
u_gen, mejor_fitness, log = run_genetico(
    R_obs, Delta_S_obs, S_inicial, S_max=S_MAX,
    n_pop=500, n_gen=200, verbose=False)
t_gen = time.time() - t0

srs_gen, S_gen_traj = calcular_srs(u_gen, R_obs, Delta_S_obs, S_inicial, S_MAX)
resultados['Baseline 2 — Genético (DEAP)'] = {'srs': srs_gen, 'u': u_gen,
                                                'S_traj': S_gen_traj, 't': t_gen}
print(f'✅ Genético completado en {t_gen:.1f}s')
print(f'   SRS = {srs_gen:.6f}  ΔSRS = {srs_gen-srs_hist:+.6f}')
print(f'   u(t): {np.round(u_gen/1000, 1)} miles TCM')

## Celda 5 — Construcción de la Matriz QUBO

El problema de liberación discreta se formula como:

$$\min_{x \in \{0,1\}^{5T}} x^\top Q x$$

donde cada $x_t = (x_{t,0}, \ldots, x_{t,4})$ es one-hot y $u(t) = \Delta u \cdot \sum_k a_k x_{t,k}$

La matriz $Q$ incluye 6 componentes:
- $Q_{dev}$: Minimizar desviación de liberaciones
- $Q_{smooth}$: Minimizar variación semana a semana  
- $Q_{crit}$: Penalizar caída bajo $S_{min}$ (factor principal)
- $Q_{onehot}$: Restricción one-hot por semana
- $Q_{flujo}$: Restricción $R(t) \geq 0$
- $Q_{balance}$: Restricción $|\sum u| \leq \eta \cdot \sum R_{obs}$

In [ ]:
from qubo_solver import construir_qubo

print('⚛️  Construyendo matriz QUBO...')
Q, params = construir_qubo(R_obs, Delta_S_obs, S_inicial, S_MAX)

n_vars = L * T
print(f'\n📐 Dimensión Q: {n_vars}×{n_vars}')
print(f'   Variables binarias: {n_vars}')
print(f'   Combinaciones posibles: L^T = 5^{T} ≈ {5**T:.2e}')
print(f'   w1={params["w1"]:.2e}  w2={params["w2"]:.2e}  w3={params["w3"]:.2e}')

# Visualizar estructura esparsidad de Q
fig, ax = plt.subplots(figsize=(7,6), facecolor='#1a1d27')
ax.set_facecolor('#0f1117')
Qlog = np.log1p(np.abs(Q))
im = ax.imshow(Qlog, cmap='plasma', aspect='auto')
plt.colorbar(im, ax=ax, label='log(1+|Q_ij|)', shrink=0.8)
ax.set_title(f'Estructura de la Matriz QUBO ({n_vars}×{n_vars})',
             color='white', fontweight='bold')
ax.set_xlabel('Variable j', color='#aaaaaa')
ax.set_ylabel('Variable i', color='#aaaaaa')
ax.tick_params(colors='#aaaaaa')
plt.tight_layout()
# Asegurar carpeta de salida antes de guardar figuras.
os.makedirs('resultados', exist_ok=True)
plt.savefig('resultados/qubo_matrix.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('📄 Guardado: resultados/qubo_matrix.png')


## Celda 6 — Solver SA en CPU (baseline cuántico)

In [ ]:
from qubo_solver import resolver_qubo_sa, decodificar_solucion

print('💻 SA CPU (8 restarts × 80,000 iter)...')
t0 = time.time()
x_cpu, E_cpu = resolver_qubo_sa(
    Q, T, R_obs=R_obs, Delta_S_obs=Delta_S_obs,
    S_inicial=S_inicial, S_max=S_MAX,
    n_restarts=8, n_iter=80_000, seed=42)
t_cpu = time.time() - t0

u_qubo_cpu = decodificar_solucion(x_cpu, T, delta_u)
srs_cpu, S_cpu_traj = calcular_srs(u_qubo_cpu, R_obs, Delta_S_obs, S_inicial, S_MAX)
resultados['QUBO + SA (CPU)'] = {'srs': srs_cpu, 'u': u_qubo_cpu,
                                  'S_traj': S_cpu_traj, 't': t_cpu}
print(f'✅ SA CPU completado en {t_cpu:.1f}s')
print(f'   SRS = {srs_cpu:.6f}  ΔSRS = {srs_cpu-srs_hist:+.6f}')

## Celda 7 — Seleccion de modo QCentroid

QCentroid no siempre renderiza botones `ipywidgets`. Por eso esta celda usa un selector por variable, que es mas confiable.

Antes de ejecutar la celda, cambia esta linea:

```python
MODO_QCENTROID_ELEGIDO = 'sa_gpu'
```

Notas para elegir:

| Modo | Cuando usarlo | Ventaja | Cuidado |
|---|---|---|---|
| `'sa_gpu'` | Entrega oficial y corrida principal | Rapido, estable, recomendado para `T=26/L=5` | Es hibrido/clasico, no QAOA puro |
| `'qaoa'` | Demo cuantica/variacional | Muestra la ruta QAOA sobre QUBO/Ising | Para `T=26` son 130 qubits; puede tardar o hacer fallback |
| `'sa_cpu'` | Validar sin GPU | Funciona aunque no haya GPU | Mas lento |

Si eliges `'qaoa'`, puedes cambiar tambien:

```python
FORZAR_QAOA_130_QUBITS = False
```

- `False`: modo protegido; si QAOA es demasiado grande, cae a `sa_gpu`.
- `True`: intenta QAOA de 130 qubits bajo tu responsabilidad.


In [ ]:
from qcentroid_solver import run, MockUseCase

use_case = MockUseCase(R_obs, Delta_S_obs, S_inicial)

# ==========================================================
# SELECTOR ROBUSTO PARA QCENTROID
# ==========================================================
# Cambia SOLO esta variable antes de ejecutar la celda.
#
# OPCION 1: 'sa_gpu'
#   Usalo para la entrega oficial. Es el modo recomendado.
#   Pros: rapido, estable, ideal para T=26/L=5.
#   Cuidado: es annealing hibrido/clasico, no QAOA puro.
#
# OPCION 2: 'qaoa'
#   Usalo si quieres intentar o demostrar la ruta cuantica/variacional.
#   Pros: conecta la formulacion QUBO/Ising con QAOA.
#   Cuidado: T=26/L=5 implica 130 qubits; puede tardar, fallar o hacer fallback.
#
# OPCION 3: 'sa_cpu'
#   Usalo si no hay GPU o quieres validar que todo corre.
#   Pros: no requiere GPU.
#   Cuidado: es mas lento.
MODO_QCENTROID_ELEGIDO = 'sa_gpu'

# Solo aplica si MODO_QCENTROID_ELEGIDO = 'qaoa'.
# False = protegido; puede hacer fallback a sa_gpu si son demasiados qubits.
# True  = intenta QAOA de 130 qubits bajo tu responsabilidad.
FORZAR_QAOA_130_QUBITS = False

print('🧭 Modo elegido:', MODO_QCENTROID_ELEGIDO)
print('')
print('Notas de seleccion:')
print('  sa_gpu → recomendado para entrega: rapido, estable, T=26/L=5.')
print('  qaoa   → ruta cuantica/variacional: 130 qubits en T=26; puede hacer fallback.')
print('  sa_cpu → fallback sin GPU: mas lento, pero robusto.')
print('')
print('Cambia MODO_QCENTROID_ELEGIDO antes de correr esta celda si quieres otro modo.')


def _params_qcentroid(modo, force_qaoa=False):
    params = {
        'backend': 'nvidia' if CUPY else 'qpp-cpu',
        'modo': modo,
        'T': T,
    }
    if modo == 'sa_gpu':
        params.update({
            'backend': 'nvidia',
            'n_restarts': 128 if CUPY else 8,
            'n_iter': 500_000 if CUPY else 80_000,
        })
    elif modo == 'sa_cpu':
        params.update({
            'backend': 'qpp-cpu',
            'n_restarts': 8,
            'n_iter': 80_000,
        })
    elif modo == 'qaoa':
        params.update({
            'backend': 'nvidia' if CUDAQ else 'qpp-cpu',
            'n_layers': 2,
            'force_qaoa': bool(force_qaoa),
            'qaoa_max_qubits': 130 if force_qaoa else 24,
            # Si force_qaoa=False y T=26, qcentroid_solver hara fallback a sa_gpu.
            'n_restarts': 128 if CUPY else 8,
            'n_iter': 500_000 if CUPY else 80_000,
        })
    else:
        raise ValueError("Modo no reconocido. Usa 'sa_gpu', 'qaoa' o 'sa_cpu'.")
    return params


def ejecutar_modo_qcentroid(modo='sa_gpu', force_qaoa=False, registrar=True):
    print('='*72)
    print(f'  Ejecutando QCentroid solver | modo={modo} | force_qaoa={force_qaoa}')
    print('='*72)
    params = _params_qcentroid(modo, force_qaoa=force_qaoa)
    print('Parametros:', params)
    resultado = run(use_case, params)

    u_sel = np.array(resultado['u_opt'], dtype=float)
    srs_sel, S_sel = calcular_srs(u_sel, R_obs, Delta_S_obs, S_inicial, S_MAX)
    nombre = f'QCentroid — {modo}'
    fallback = resultado.get('meta', {}).get('fallback_from')
    if fallback:
        nombre += ' (fallback a sa_gpu)'

    if registrar:
        resultados[nombre] = {
            'srs': srs_sel,
            'u': u_sel,
            'S_traj': S_sel,
            't': resultado.get('tiempo_s', None),
            'meta': resultado.get('meta', {}),
        }

        global u_qubo_gpu, srs_gpu, S_gpu_traj, resultado_qcentroid
        u_qubo_gpu = u_sel
        srs_gpu = srs_sel
        S_gpu_traj = S_sel
        resultado_qcentroid = resultado

        if modo in ('sa_gpu', 'qaoa') and (fallback or modo == 'sa_gpu'):
            resultados['QUBO + SA (GPU)'] = resultados[nombre].copy()
        elif modo == 'sa_cpu':
            resultados['QUBO + SA (CPU)'] = resultados[nombre].copy()

        os.makedirs('resultados', exist_ok=True)
        with open('resultados/qcentroid_resultado.json', 'w') as f:
            json.dump(resultado, f, indent=2)
        print('📄 Guardado: resultados/qcentroid_resultado.json')

    print(f'✅ Resultado {nombre}')
    print(f'   SRS = {srs_sel:.6f}  ΔSRS = {srs_sel-srs_hist:+.6f}')
    print(f'   Tiempo = {resultado.get("tiempo_s", 0):.1f}s')
    print(f'   Meta = {resultado.get("meta", {})}')
    print('ℹ️ Si quieres que este resultado aparezca en tabla/graficas, ejecuta las celdas siguientes.')
    return resultado


resultado_qcentroid = ejecutar_modo_qcentroid(
    MODO_QCENTROID_ELEGIDO,
    force_qaoa=FORZAR_QAOA_130_QUBITS,
    registrar=True,
)


## Celda 9 — Tabla Benchmark ΔSRS Oficial

In [ ]:
print('='*75)
print(f'  📊 BENCHMARK ΔSRS OFICIAL — Embalse Falcón (T={T}, L={L})')
print('='*75)
print(f'  {"Método":<40} {"SRS":>10}  {"ΔSRS":>12}  {"Tiempo":>8}')
print('  ' + '-'*72)

rows = []
for nombre, data in resultados.items():
    srs_val = data['srs']
    delta   = srs_val - srs_hist
    tiempo  = f"{data.get('t', 0):.1f}s" if 't' in data else '  —'
    marca   = '—' if abs(delta) < 1e-10 else (f'+{delta:.6f} ✅' if delta > 0 else f'{delta:.6f} ⚠️')
    print(f'  {nombre:<40} {srs_val:>10.6f}  {marca:>12}  {tiempo:>8}')
    rows.append({'Metodo': nombre, 'SRS': srs_val, 'ΔSRS': delta,
                 'Tiempo_s': data.get('t', None)})

print('='*75)

# Guardar CSV
os.makedirs('resultados', exist_ok=True)
pd.DataFrame(rows).to_csv('resultados/benchmark_delta_srs.csv', index=False)
print('\n📄 Guardado: resultados/benchmark_delta_srs.csv')

## Celda 10 — Gráfica Benchmark (Almacenamiento + u(t))

In [ ]:
fechas = pd.date_range(start=FECHA, periods=T+1, freq='W-SUN')

# Histórico real (simulado sin ajuste)
S_hist_real = np.zeros(T+1)
S_hist_real[0] = S_inicial
for i in range(T):
    S_hist_real[i+1] = S_hist_real[i] + Delta_S_obs[i]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.06})
fig.patch.set_facecolor('#0f1117')
for ax in (ax1, ax2): ax.set_facecolor('#1a1d27')

# Panel 1: Almacenamiento
colores = {'Histórico': '#cccccc', 'Regla': '#facc15',
           'Genético': '#38bdf8', 'CPU': '#a78bfa', 'GPU': '#34d399'}

ax1.fill_between(fechas, S_hist_real/1000, alpha=0.10, color='#cccccc')
ax1.plot(fechas, S_hist_real/1000, color='#cccccc', lw=2, label='Histórico obs.', zorder=3)

ax1.plot(fechas, np.concatenate([[S_inicial], resultados['Baseline 1 — Regla umbral']['S_traj'][:-1]])/1000,
         color='#facc15', lw=1.6, ls=':', label='Baseline 1 — Regla umbral', zorder=4)
ax1.plot(fechas, np.concatenate([[S_inicial], resultados['Baseline 2 — Genético (DEAP)']['S_traj'][:-1]])/1000,
         color='#38bdf8', lw=2.4, ls='--',
         label=f'Baseline 2 — Genético  (ΔSRS={resultados["Baseline 2 — Genético (DEAP)"]["srs"]-srs_hist:+.4f})', zorder=5)
ax1.plot(fechas, np.concatenate([[S_inicial], resultados['QUBO + SA (CPU)']['S_traj'][:-1]])/1000,
         color='#a78bfa', lw=2, ls='-.',
         label=f'QUBO + SA (CPU)  (ΔSRS={resultados["QUBO + SA (CPU)"]["srs"]-srs_hist:+.4f})', zorder=5)

if 'QUBO + SA (GPU)' in resultados and CUPY:
    ax1.plot(fechas, np.concatenate([[S_inicial], resultados['QUBO + SA (GPU)']['S_traj'][:-1]])/1000,
             color='#34d399', lw=2.6, ls='solid',
             label=f'QUBO + SA (GPU)  (ΔSRS={resultados["QUBO + SA (GPU)"]["srs"]-srs_hist:+.4f})', zorder=6)

ax1.axhline(NAMO/1000,  color='#60a5fa', ls='--', lw=1.2, alpha=0.7, label=f'NAMO ({NAMO/1000:,.0f}k TCM)')
ax1.axhline(S_MIN/1000, color='#f87171', ls=':',  lw=2.0, alpha=0.95, label=f'Crítico 25% ({S_MIN/1000:,.0f}k TCM)')
ax1.axhspan(0, S_MIN/1000, alpha=0.07, color='#f87171', zorder=0)
ax1.text(fechas[1], S_MIN/1000 * 0.91, '⚠ Zona crítica (<25% NAMO)', color='#f87171', fontsize=8.5, alpha=0.9)

S_all = S_hist_real.copy()
ax1.set_ylim(max(0, S_all.min()/1000 * 0.86), S_all.max()/1000 * 1.12)
ax1.set_ylabel('Almacenamiento (miles TCM)', color='#aaaaaa', fontsize=11)
ax1.set_title('Benchmark Comparativo — Embalse Falcón  |  T=26 semanas, L=5 niveles',
              color='white', fontsize=13, fontweight='bold', pad=10)
ax1.legend(loc='lower right', fontsize=9, framealpha=0.4, facecolor='#1a1d27', labelcolor='white')
ax1.tick_params(colors='#aaaaaa')
ax1.grid(True, ls='--', alpha=0.15, color='white')

# Panel 2: u(t)
qubo_plot_data = resultados.get('QUBO + SA (GPU)', resultados['QUBO + SA (CPU)'])
u_qubo_plot = qubo_plot_data['u']
fechas_u = fechas[:-1]
ancho = 3.5
ax2.bar(fechas_u - pd.Timedelta(days=ancho*1.5), resultados['Baseline 1 — Regla umbral']['u']/1000,
        width=ancho, color='#facc15', alpha=0.75, label='Regla', align='center')
ax2.bar(fechas_u - pd.Timedelta(days=ancho*0.5), resultados['Baseline 2 — Genético (DEAP)']['u']/1000,
        width=ancho, color='#38bdf8', alpha=0.85, label='Genético', align='center')
ax2.bar(fechas_u + pd.Timedelta(days=ancho*0.5), u_qubo_plot/1000,
        width=ancho, color='#34d399', alpha=0.85, label='QUBO+SA', align='center')

ax2.axhline(0, color='white', lw=0.8, alpha=0.4)
ax2.axhline(-delta_u/1000, color='#f87171', ls=':', lw=1.0, alpha=0.5)
ax2.axhline( delta_u/1000, color='#60a5fa', ls=':', lw=1.0, alpha=0.5)
ax2.text(fechas_u[0], -delta_u/1000 * 1.35,
         '← u<0: reducir liberación (embalse se llena)', color='#f87171', fontsize=7.5, alpha=0.85)
ax2.text(fechas_u[0],  delta_u/1000 * 1.10,
         '→ u>0: aumentar liberación (embalse se vacía)', color='#60a5fa', fontsize=7.5, alpha=0.85)
ax2.set_ylabel('u(t) (miles TCM/sem)', color='#aaaaaa', fontsize=10)
ax2.set_xlabel('Fecha', color='#aaaaaa', fontsize=11)
ax2.legend(loc='lower right', fontsize=8, framealpha=0.35, facecolor='#1a1d27', labelcolor='white')
ax2.tick_params(colors='#aaaaaa')
ax2.grid(True, ls='--', alpha=0.12, color='white')

for ax in (ax1, ax2):
    for spine in ax.spines.values(): spine.set_edgecolor('#444444')

plt.tight_layout()
# Asegurar carpeta de salida antes de guardar figuras.
os.makedirs('resultados', exist_ok=True)
plt.savefig('resultados/simulacion_hibrida.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('📄 Guardado: resultados/simulacion_hibrida.png')


## Celda 11 — Visualizar Matriz QUBO y Justificación Cuántica

In [ ]:
# ── Gráfica de energía Q por componente ──────────────────────────────────────
from qubo_solver import (q_dev, q_smooth, q_crit, q_onehot, q_flujo, q_balance)

w1 = params['w1']; w2 = params['w2']; w3 = params['w3']
S_min_val = params['S_min']
lam_oh    = params['lam_onehot']
lam_r     = params['lam_R']
lam_bal   = params['lam_bal']

componentes = {
    'Q_dev'    : q_dev(T, delta_u, w2),
    'Q_smooth' : q_smooth(T, delta_u, w3),
    'Q_crit'   : q_crit(T, delta_u, w1, Delta_S_obs, S_inicial, S_min_val),
    'Q_onehot' : q_onehot(T, lam_oh),
    'Q_flujo'  : q_flujo(T, R_obs, delta_u, lam_r),
    'Q_balance': q_balance(T, delta_u, lam_bal),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9), facecolor='#0f1117')
axes = axes.flatten()
for idx, (nombre, Qc) in enumerate(componentes.items()):
    ax = axes[idx]
    ax.set_facecolor('#1a1d27')
    im = ax.imshow(np.log1p(np.abs(Qc)), cmap='viridis', aspect='auto')
    ax.set_title(nombre, color='white', fontsize=11, fontweight='bold')
    ax.tick_params(colors='#aaaaaa', labelsize=7)
    plt.colorbar(im, ax=ax, shrink=0.8)
    for spine in ax.spines.values(): spine.set_edgecolor('#444444')

fig.suptitle('Componentes de la Matriz QUBO — log(1+|Q_ij|)', 
             color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
# Asegurar carpeta de salida antes de guardar figuras.
os.makedirs('resultados', exist_ok=True)
# Asegurar carpeta de salida antes de guardar figuras.
os.makedirs('resultados', exist_ok=True)
plt.savefig('resultados/qubo_components.png', dpi=130, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

# ── Justificación cuántica ────────────────────────────────────────────────────
print('\n📐 JUSTIFICACIÓN DE COMPUTACIÓN CUÁNTICA:')
print('─'*60)
for inst in [('Small', 12, 3), ('Medium (oficial)', 26, 5), ('Large', 52, 5)]:
    nombre, t_val, l_val = inst
    comb = l_val ** t_val
    print(f'  {nombre:<20} T={t_val:>3} L={l_val}  → {l_val}^{t_val} = {comb:.2e} combinaciones')
print('─'*60)
print('  La instancia Medium tiene más combinaciones que átomos en el universo observable.')
print('  Los solvers cuánticos explotan la superposición para evaluar múltiples')
print('  estados simultáneamente, evitando la búsqueda exhaustiva.')


## Celda 12 — Integración con QCentroid SDK

In [ ]:
# ── Modo QCentroid SDK (opcional, solo si tienes API key) ─────────────────────

USAR_QCENTROID_SDK = False   # Cambiar a True si tienes acceso al SDK
API_KEY = 'TU_API_KEY_AQUI'

if USAR_QCENTROID_SDK:
    from qcentroid_sdk import QCentroidClient
    client = QCentroidClient(api_key=API_KEY)

    dataset_id = client.upload_dataset({
        'R_obs'       : R_obs.tolist(),
        'Delta_S_obs' : Delta_S_obs.tolist(),
        'S_inicial'   : float(S_inicial),
    })
    print(f'✅ Dataset subido: {dataset_id}')

    # Usa el mismo modo elegido en el notebook. Por defecto: sa_gpu.
    solver_params_sdk = _params_qcentroid(
        MODO_QCENTROID_DEFAULT,
        force_qaoa=FORZAR_QAOA_130_QUBITS,
    )

    job = client.run_job(
        solver_file='qcentroid_solver.py',
        dataset_id=dataset_id,
        solver_params=solver_params_sdk,
    )
    print('⏳ Esperando resultado...')
    resultado_sdk = job.wait_for_result()
    print(f'✅ SRS obtenido: {resultado_sdk["SRS"]:.6f}')
    print(f'   Tiempo: {resultado_sdk["tiempo_s"]:.1f}s')
else:
    print('ℹ️ SDK de QCentroid desactivado.')
    if 'resultado_qcentroid' in globals():
        os.makedirs('resultados', exist_ok=True)
        with open('resultados/qcentroid_resultado.json', 'w') as f:
            json.dump(resultado_qcentroid, f, indent=2)
        print('📄 Guardado: resultados/qcentroid_resultado.json')
    else:
        print('ℹ️ Aun no se ha ejecutado el boton del selector QCentroid en la celda 7.')


## Celda 13 — Resumen final y exportación

In [ ]:
print('═'*70)
print('  RESUMEN FINAL — Challenge A: Falcon Reservoir')
print('  Hackathon Latinoamérica Cuántico 2026')
print('═'*70)
print(f'  Instancia: T={T} semanas, L={L} niveles, {L**T:.2e} combinaciones')
print(f'  S_inicial = {S_inicial:,.0f} TCM ({S_inicial/NAMO*100:.1f}% NAMO)  →  Estado crítico')
print()

mejor_srs = max(r['srs'] for r in resultados.values())
mejor_met = [k for k, v in resultados.items() if v['srs'] == mejor_srs][0]

print(f'  📊 ΔSRS respecto al baseline histórico:')
for nombre, data in resultados.items():
    delta = data['srs'] - srs_hist
    marca = '⭐ MEJOR' if data['srs'] == mejor_srs else ''
    print(f'     {nombre:<40} {delta:>+.6f}  {marca}')

print()
print(f'  🏆 Mejor método: {mejor_met}')
print(f'     ΔSRS = {mejor_srs - srs_hist:+.6f}')
print()
print('  📂 Archivos generados:')
if os.path.isdir('resultados'):
    archivos_resultados = os.listdir('resultados')
else:
    archivos_resultados = []
for f in archivos_resultados:
    size = os.path.getsize(f'resultados/{f}')
    print(f'     resultados/{f}  ({size/1024:.1f} KB)')
print('═'*70)
